In [ ]:
import csv
from itertools import combinations
from collections import Counter

In [ ]:
DATA_FOLDER = "../data/"
CSV_ACTORS_PER_PARAGAPH_PER_YEAR = f"{DATA_FOLDER}actors_per_paragraph_per_year.csv"
CSV_EDGES_COOCCURRENCES_NETWORK = f"{DATA_FOLDER}edges_cooccurrence_network.csv"
LIST_OF_YEARS = ["1996", "1997", "1998", "1999", "2000", "2001", "2002", "2003", "2005", "2006", "2007", "2024"]

In [ ]:
import csv
from itertools import combinations

def get_actor_cooccurrences_by_paragraph_for_year(year: str) -> list[tuple[str, str, int]]:
    print(f"Getting actor cooccurrences for year {year}...")
    actor_pairs_count = {}

    with open(CSV_ACTORS_PER_PARAGAPH_PER_YEAR, mode='r', encoding="utf-8") as file:
        csvFile = csv.reader(file)
        next(csvFile, None)  # skip the headers

        for lines in csvFile:
            if lines[0] == year:
                actors_array = lines[2].split(",")
                actor_set = set()

                for actor in actors_array:
                    actor_set.add(actor.strip())

                for actor_tuple in combinations(actor_set, 2):
                    source = actor_tuple[0]
                    target = actor_tuple[1]

                    if source < target:
                        normalized_tuple = (source, target)
                    else:
                        normalized_tuple = (target, source)

                    if normalized_tuple not in actor_pairs_count:
                        actor_pairs_count[normalized_tuple] = 1
                    else:
                        actor_pairs_count[normalized_tuple] += 1

    actor_pairs = []

    for normalized_tuple in actor_pairs_count:
        source = normalized_tuple[0]
        target = normalized_tuple[1]
        weight = actor_pairs_count[normalized_tuple]
        actor_pairs.append((source, target, weight))

    print(f"Found {len(actor_pairs)} unique cooccurrences for year {year}.")
    print(actor_pairs)
    return actor_pairs

In [ ]:
def build_csv_edges_coocurrence_network(edges: list[tuple[str, str, int]], year: str):
    print(f"Building CSV edges cooccurrence network for year {year}...")
    with open(CSV_EDGES_COOCCURRENCES_NETWORK, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        for actors_tuple in edges:
            writer.writerow([actors_tuple[0], actors_tuple[1], actors_tuple[2], year, year])
    print(f"Finished building CSV edges cooccurrence network for year {year}.")

In [ ]:
with open(CSV_EDGES_COOCCURRENCES_NETWORK, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(['Source', 'Target', 'Weight', 'Start', 'End'])

for year in LIST_OF_YEARS:
    print(f"Processing year {year}...")
    edges = get_actor_cooccurrences_by_paragraph_for_year(year)
    build_csv_edges_coocurrence_network(edges, year)
    print(f"Finished processing year {year}.")